In [1]:
import os
import sys
import glob
import json
import h5py
import warnings
import numpy as np
import pandas as pd
from pathlib import Path
from datetime import datetime
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

from vip_slap2_analysis.io.session_registry import VIPSessionRegistry
from vip_slap2_analysis.voltage.extraction import load_voltage_roi_transform_h5
from vip_slap2_analysis.voltage import analysis
from vip_slap2_analysis.common.qc import robust_sigma
from vip_slap2_analysis.plotting.plot_psth import plot_voltage_mean_image_response_heatmap
from vip_slap2_analysis.utils.utils import save_figure

import matplotlib.pyplot as plt
from IPython.display import display, HTML

import seaborn as sns
sns.set_style('white')
params = {'legend.fontsize': 'x-large',
         'axes.labelsize': 'xx-large',
         'axes.titlesize':'xx-large',
         'xtick.labelsize':'xx-large',
         'ytick.labelsize':'xx-large'}
plt.rcParams.update(params)

display(HTML("<style>.container { width:100% !important; }</style>"))

In [2]:
%load_ext autoreload
%autoreload 2

%matplotlib notebook

## Build session registry

In [3]:
today_str = datetime.today().strftime('%Y-%m-%d')

BASE_PATH = Path(r'\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics')
SAVE_PATH = Path(r'C:\Users\andrew.shelton\Dropbox\allen institute\Documents\Presentations\OPhys\Lab_Meetings\2026-07-28_OPhys_LabMeetingV\figures\voltage_plots')

TARGET_MICE = [
    826031,
    826032
]

PARADIGMS = ["change_detection_passive"]
EXCLUDE_SESSION_TYPES = ["expression_check", "volume_imaging"]

# Voltage extraction writes files like:
#   voltage_session_traces_dff_robust_f0_trial.h5
# This variant controls the filename suffix. The plotted dataset is SIGNAL below.
TRACE_VARIANT = "dff_robust_f0_trial"
SIGNAL = "dff"      # one of: "raw_f", "f0", "dff"

# Optional direct override. Leave as None to resolve from asset.derived_dir / "voltage".
SESSION_TRACE_H5 = None

In [4]:
registry = VIPSessionRegistry.from_basepath(BASE_PATH)

process_df = registry.sessions(
    subject_ids=TARGET_MICE,
    exclude_session_types=EXCLUDE_SESSION_TYPES,
    paradigms=PARADIGMS,
)

assets = [registry.resolve_assets(row) for _, row in process_df.iterrows()]

print(f"Found {len(assets)} candidate sessions.")
display(process_df)

Found 19 candidate sessions.


,session_id,subject_id,session_#,session_date,indicator1,indicator2,dmd1_depth,dmd2_depth,paradigm,session_type,...,instrument_id,camera_type,has raster ROI?,has integration roi?,behavior_rig,quality,flags,session_dir,purpose,notes
0,826031_2026-01-30_15-04-02,826031,2,2026-01-30,ASAP7y,NaN,25,250,change_detection_passive,familiar,...,SLAP2_1,spinnaker,yes,yes,VCO.1,good,NaN,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...,NaN,NaN
1,826031_2026-02-01_11-01-50,826031,3,2026-02-01,ASAP7y,NaN,25,250,change_detection_passive,familiar,...,SLAP2_1,spinnaker,yes,yes,VCO.1,good,NaN,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...,NaN,NaN
2,826031_2026-02-02_10-23-53,826031,4,2026-02-02,ASAP7y,NaN,25,250,change_detection_passive,familiar,...,SLAP2_1,spinnaker,yes,yes,VCO.1,good,NaN,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...,NaN,NaN
3,826031_2026-02-03_14-21-45,826031,5,2026-02-03,ASAP7y,NaN,25,250,change_detection_passive,familiar,...,SLAP2_1,spinnaker,yes,yes,VCO.1,good,NaN,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...,NaN,NaN
4,826031_2026-02-04_12-15-34,826031,6,2026-02-04,ASAP7y,NaN,25,250,change_detection_passive,familiar,...,SLAP2_1,spinnaker,yes,yes,VCO.1,good,NaN,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...,NaN,NaN
5,826031_2026-02-05_09-28-56,826031,7,2026-02-05,ASAP7y,NaN,25,250,change_detection_passive,familiar,...,SLAP2_1,spinnaker,yes,yes,VCO.1,good,NaN,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...,NaN,NaN
6,826031_2026-02-06_10-21-20,826031,8,2026-02-06,ASAP7y,NaN,25,250,change_detection_passive,familiar,...,SLAP2_1,spinnaker,yes,yes,VCO.1,good,NaN,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...,NaN,NaN
7,826031_2026-02-10_10-48-51,826031,9,2026-02-10,ASAP7y,NaN,25,250,change_detection_passive,novel,...,SLAP2_1,spinnaker,yes,yes,VCO.1,good,NaN,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...,NaN,NaN
8,826031_2026-02-11_12-42-21,826031,10,2026-02-11,ASAP7y,NaN,25,250,change_detection_passive,novel+,...,SLAP2_1,spinnaker,yes,yes,VCO.1,good,NaN,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...,NaN,NaN
9,826031_2026-02-12_07-43-37,826031,11,2026-02-12,ASAP7y,NaN,25,250,change_detection_passive,novel+,...,SLAP2_1,spinnaker,yes,yes,VCO.1,good,NaN,\\allen\aind\scratch\ophys\Andrew\VIP_synaptic...,NaN,NaN



## Frequency-domain characterization of ASAP7y VIP dendritic dF/F

This notebook preserves the registry/asset-loading scheme from the stub and adds slide-oriented frequency analyses for full-session voltage traces.

Main outputs:

1. **ROI × frequency PSD heatmaps** grouped by DMD and sorted by spectral centroid.
2. **ROI × bandpower heatmaps** for interpretable frequency bands.
3. **Bandpower distributions by DMD** across all sessions.
4. **Frequency phenotype scatter**: spectral centroid × spectral entropy, optionally sized by SNR from the companion SNR notebook.
5. **Image-cadence enrichment** around 1.33 Hz and harmonics.
6. **Example frequency phenotypes**: slow-dominated, task-band-enriched, and broadband/fast ROIs.
7. **Session-level summaries** to reduce pseudoreplication when comparing DMD/depth.

PSD estimates are computed from evenly distributed windows, anti-aliased/downsampled before Welch estimation, rather than by loading entire multi-gigabyte traces into memory.


In [5]:

from scipy import signal

# -----------------------------
# Example-session display
# -----------------------------
EXAMPLE_ASSET_INDEX = 0
TRACE_START_SEC = 120          # None -> begin 30 s after trace start
TRACE_DURATION_SEC = 30
TRACE_DISPLAY_FS_HZ = 1000     # display traces only; PSD uses DOWNSAMPLE_TARGET_HZ

# -----------------------------
# PSD estimation
# -----------------------------
PSD_WINDOW_SEC = 120.0          # long enough to resolve ~0.01-0.1 Hz components
N_PSD_WINDOWS = 4               # distributed across each recording
PAD_SEC = 30.0                  # avoid acquisition edges when possible
PSD_MAX_HZ = 500.0
DOWNSAMPLE_TARGET_HZ = 1200.0   # actual fs = native fs / integer factor, with Nyquist > PSD_MAX_HZ
WELCH_SEGMENT_SEC = 64.0        # frequency resolution ~1 / segment duration
WELCH_OVERLAP = 0.5
WELCH_AVERAGE = 'median'        # robust if supported by your SciPy; otherwise falls back to mean
MIN_VALID_SAMPLES = 1000

# Frequency plotting / interpolation
FREQ_MIN_PLOT = 0.01
FREQ_MAX_PLOT = 500.0
N_LOG_FREQ_BINS = 240
PSD_EPS = 1e-24
HEATMAP_ZLIM = 1.5
MAX_HEATMAP_ROWS = None         # set e.g. 250 if the all-session heatmap is too dense

# Passive DoC image cadence: 250 ms image + 500 ms gray = 0.75 s period.
IMAGE_CADENCE_HZ = 1 / 0.75
N_CADENCE_HARMONICS = 6
CADENCE_HALF_BAND_HZ = 0.04
CADENCE_BACKGROUND_HZ = 0.6
CADENCE_EXCLUSION_HZ = 0.08

FREQUENCY_BANDS = {
    'very_slow_0p01_0p1': (0.01, 0.1),
    'slow_0p1_0p5': (0.1, 0.5),
    'task_0p5_2': (0.5, 2.0),
    'intermediate_2_10': (2.0, 10.0),
    'fast_10_100': (10.0, 100.0),
    'hf_100_500': (100.0, 500.0),
}
BAND_LABELS = {
    'very_slow_0p01_0p1': 'Very slow\n0.01-0.1 Hz',
    'slow_0p1_0p5': 'Slow\n0.1-0.5 Hz',
    'task_0p5_2': 'Task cadence\n0.5-2 Hz',
    'intermediate_2_10': 'Intermediate\n2-10 Hz',
    'fast_10_100': 'Fast\n10-100 Hz',
    'hf_100_500': 'HF/noise floor\n100-500 Hz',
}
POWER_RANGE_HZ = (0.01, PSD_MAX_HZ)

# Optional companion SNR merge. None = auto-search SAVE_PATH for latest '*ASAP7_dynamic_snr_all_sessions.csv'.
SNR_METRICS_CSV = None

SAVE_FIGURES = True
SAVE_TABLES = True
SAVE_PATH.mkdir(parents=True, exist_ok=True)
print(f'Figures/tables will be written to: {SAVE_PATH}')


Figures/tables will be written to: C:\Users\andrew.shelton\Dropbox\allen institute\Documents\Presentations\OPhys\Lab_Meetings\2026-07-28_OPhys_LabMeetingV\figures\voltage_plots


## Helper functions

In [6]:

def resolve_trace_h5(asset, trace_variant=TRACE_VARIANT, override=SESSION_TRACE_H5):
    if override is not None:
        path = Path(override)
    else:
        path = Path(asset.derived_dir) / 'voltage' / f'voltage_session_traces_{trace_variant}.h5'
    if not path.exists():
        raise FileNotFoundError(path)
    return path


def _decode_roi_ids(values):
    out = []
    for value in np.asarray(values):
        if isinstance(value, (bytes, np.bytes_)):
            value = value.decode(errors='replace')
        out.append(str(value))
    return np.asarray(out, dtype=object)


def _trace_layout(dataset, n_time):
    if dataset.ndim != 2:
        raise ValueError(f'Expected a 2-D trace dataset, found shape {dataset.shape}')
    if dataset.shape[1] == n_time:
        return 'roi_time'
    if dataset.shape[0] == n_time:
        return 'time_roi'
    raise ValueError(f'Neither trace axis matches timebase length {n_time}: {dataset.shape}')


def _read_trace_block(dataset, i0, i1, layout):
    if layout == 'roi_time':
        return np.asarray(dataset[:, i0:i1], dtype=np.float64)
    return np.asarray(dataset[i0:i1, :], dtype=np.float64).T


def _infer_sampling_rate_hz(timebase_sec):
    dt = np.diff(np.asarray(timebase_sec, dtype=np.float64))
    dt = dt[np.isfinite(dt) & (dt > 0)]
    if dt.size == 0:
        raise ValueError('Could not infer sampling rate from timebase_sec')
    return float(1.0 / np.median(dt))


def _choose_integer_decimation(fs, target_fs=DOWNSAMPLE_TARGET_HZ, max_hz=PSD_MAX_HZ):
    q = max(1, int(np.floor(fs / target_fs)))
    while q > 1 and (fs / q) / 2 <= max_hz:
        q -= 1
    if (fs / q) / 2 <= max_hz:
        warnings.warn(
            f'Native sampling rate {fs:.1f} Hz gives Nyquist {(fs/q)/2:.1f} Hz; '
            f'PSD_MAX_HZ={max_hz:.1f} may be too high.'
        )
    return q, float(fs / q)


def _distributed_starts(n_time, fs, window_sec=PSD_WINDOW_SEC, n_windows=N_PSD_WINDOWS, pad_sec=PAD_SEC):
    win = int(round(window_sec * fs))
    win = max(2, min(win, int(n_time)))
    if n_time <= win:
        return np.asarray([0], dtype=int), win
    pad = int(round(pad_sec * fs))
    lo = min(pad, n_time - win)
    hi = max(0, n_time - win - pad)
    if hi <= lo:
        lo, hi = 0, n_time - win
    starts = np.unique(np.round(np.linspace(lo, hi, max(1, int(n_windows)))).astype(int))
    starts = starts[(starts >= 0) & (starts + win <= n_time)]
    if starts.size == 0:
        starts = np.asarray([0], dtype=int)
    return starts, win


def _fill_and_center_rows(X):
    X = np.asarray(X, dtype=np.float64)
    out = np.empty_like(X)
    finite_counts = np.zeros(X.shape[0], dtype=int)
    for r in range(X.shape[0]):
        x = X[r]
        finite = np.isfinite(x)
        finite_counts[r] = int(finite.sum())
        if finite_counts[r] == 0:
            out[r] = 0.0
        else:
            med = float(np.nanmedian(x[finite]))
            y = x.copy()
            y[~finite] = med
            out[r] = y - med
    return out, finite_counts


def _downsample_block(X, q):
    if q <= 1:
        return X
    return signal.resample_poly(X, up=1, down=q, axis=1, padtype='line')


def _welch_psd_rows(X, fs_ds, segment_sec=WELCH_SEGMENT_SEC, overlap=WELCH_OVERLAP):
    nperseg = int(round(segment_sec * fs_ds))
    nperseg = max(128, min(nperseg, X.shape[1]))
    noverlap = int(round(nperseg * overlap))
    noverlap = min(max(0, noverlap), nperseg - 1)
    try:
        f, pxx = signal.welch(
            X, fs=fs_ds, axis=1, nperseg=nperseg, noverlap=noverlap,
            detrend='constant', scaling='density', average=WELCH_AVERAGE,
        )
    except TypeError:
        f, pxx = signal.welch(
            X, fs=fs_ds, axis=1, nperseg=nperseg, noverlap=noverlap,
            detrend='constant', scaling='density',
        )
    return np.asarray(f), np.asarray(pxx)


def _bandpower(f, pxx, lo, hi):
    m = (f >= lo) & (f < hi) & np.isfinite(f) & np.isfinite(pxx)
    if m.sum() < 2:
        return np.nan
    return float(np.trapz(pxx[m], f[m]))


def _spectral_centroid(f, pxx, lo=POWER_RANGE_HZ[0], hi=POWER_RANGE_HZ[1]):
    m = (f >= lo) & (f <= hi) & np.isfinite(pxx) & (pxx > 0)
    if not np.any(m):
        return np.nan
    w = pxx[m]
    return float(np.sum(f[m] * w) / np.sum(w))


def _spectral_edges(f, pxx, lo=POWER_RANGE_HZ[0], hi=POWER_RANGE_HZ[1], probs=(0.05, 0.5, 0.95)):
    m = (f >= lo) & (f <= hi) & np.isfinite(pxx) & (pxx > 0)
    if m.sum() < 2:
        return [np.nan] * len(probs)
    ff = f[m]
    pp = pxx[m]
    df = np.gradient(ff)
    cumulative = np.cumsum(pp * df)
    if cumulative[-1] <= 0:
        return [np.nan] * len(probs)
    cumulative = cumulative / cumulative[-1]
    return [float(np.interp(p, cumulative, ff)) for p in probs]


def _normalized_entropy(probabilities):
    p = np.asarray(probabilities, dtype=np.float64)
    p = p[np.isfinite(p) & (p > 0)]
    if p.size <= 1:
        return np.nan
    p = p / p.sum()
    return float(-np.sum(p * np.log(p)) / np.log(p.size))


def _loglog_slope(f, pxx, lo, hi):
    m = (f >= lo) & (f <= hi) & np.isfinite(pxx) & (pxx > 0)
    if m.sum() < 5:
        return np.nan
    return float(np.polyfit(np.log10(f[m]), np.log10(pxx[m]), deg=1)[0])


def _peak_over_local_background(f, pxx, center):
    peak = _bandpower(f, pxx, center - CADENCE_HALF_BAND_HZ, center + CADENCE_HALF_BAND_HZ)
    bg_mask = (
        (f >= center - CADENCE_BACKGROUND_HZ) & (f <= center + CADENCE_BACKGROUND_HZ) &
        ~((f >= center - CADENCE_EXCLUSION_HZ) & (f <= center + CADENCE_EXCLUSION_HZ)) &
        np.isfinite(pxx) & (pxx > 0)
    )
    if bg_mask.sum() < 2 or not np.isfinite(peak):
        return np.nan
    bg_power = float(np.nanmedian(pxx[bg_mask])) * (2 * CADENCE_HALF_BAND_HZ)
    if bg_power <= 0:
        return np.nan
    return float(peak / bg_power)


def _interpolate_row_normalized_logpsd(f, pxx, freq_grid):
    m = (f >= freq_grid[0]) & (f <= freq_grid[-1]) & np.isfinite(pxx) & (pxx > 0)
    if m.sum() < 2:
        return np.full(freq_grid.size, np.nan), np.full(freq_grid.size, np.nan)
    total = _bandpower(f, pxx, freq_grid[0], freq_grid[-1])
    if not np.isfinite(total) or total <= 0:
        return np.full(freq_grid.size, np.nan), np.full(freq_grid.size, np.nan)
    frac_density = np.maximum(pxx[m] / total, PSD_EPS)
    interp = np.interp(np.log10(freq_grid), np.log10(f[m]), np.log10(frac_density))
    mu = np.nanmean(interp)
    sd = np.nanstd(interp)
    z = (interp - mu) / sd if np.isfinite(sd) and sd > 0 else interp * np.nan
    return interp, z


def _session_metadata(asset):
    metadata = getattr(asset, 'metadata', {}) or {}
    subject_id = getattr(asset, 'subject_id', None)
    if subject_id is None:
        subject_id = getattr(asset, 'mouse_id', None)
    return {
        'subject_id': subject_id,
        'session_id': str(getattr(asset, 'session_id', 'unknown_session')),
        'dmd1_depth': metadata.get('dmd1_depth', np.nan),
        'dmd2_depth': metadata.get('dmd2_depth', np.nan),
    }


## PSD and metric estimation

In [7]:

def estimate_dmd_frequency_metrics(group, *, dmd='DMD?', signal_name=SIGNAL, freq_grid=None):
    if signal_name not in group:
        raise KeyError(f'{group.name} has no dataset {signal_name!r}')
    if freq_grid is None:
        freq_grid = np.geomspace(FREQ_MIN_PLOT, FREQ_MAX_PLOT, N_LOG_FREQ_BINS)

    t = np.asarray(group['timebase_sec'][:], dtype=np.float64)
    fs = _infer_sampling_rate_hz(t)
    ds = group[signal_name]
    layout = _trace_layout(ds, len(t))
    n_roi = ds.shape[0] if layout == 'roi_time' else ds.shape[1]
    roi_ids = _decode_roi_ids(group['roi_ids'][:]) if 'roi_ids' in group else np.asarray([str(i) for i in range(n_roi)], dtype=object)
    valid_rois = np.asarray(group['valid_rois_mask'][:], dtype=bool) if 'valid_rois_mask' in group else np.ones(n_roi, dtype=bool)

    q, fs_ds = _choose_integer_decimation(fs)
    starts, win = _distributed_starts(len(t), fs)

    pxx_sum = None
    f_keep = None
    finite_counts_total = np.zeros(n_roi, dtype=np.int64)
    windows_used = 0

    for i0 in starts:
        X = _read_trace_block(ds, int(i0), int(i0 + win), layout)
        X, finite_counts = _fill_and_center_rows(X)
        finite_counts_total += finite_counts
        X_ds = _downsample_block(X, q)
        f, pxx = _welch_psd_rows(X_ds, fs_ds)
        keep = (f >= 0) & (f <= PSD_MAX_HZ)
        f = f[keep]
        pxx = pxx[:, keep]
        if pxx_sum is None:
            pxx_sum = np.zeros_like(pxx, dtype=np.float64)
            f_keep = f
        elif pxx.shape != pxx_sum.shape or not np.allclose(f, f_keep):
            pxx = np.vstack([np.interp(f_keep, f, row, left=np.nan, right=np.nan) for row in pxx])
        pxx_sum += pxx
        windows_used += 1

    if windows_used == 0 or pxx_sum is None:
        raise RuntimeError(f'No PSD windows could be estimated for {group.name}')

    f = f_keep
    pxx_mean = pxx_sum / windows_used
    metric_rows = []
    psd_rows = []

    for r in range(n_roi):
        p = pxx_mean[r]
        total_power = _bandpower(f, p, POWER_RANGE_HZ[0], POWER_RANGE_HZ[1])
        band_powers = {name: _bandpower(f, p, lo, hi) for name, (lo, hi) in FREQUENCY_BANDS.items()}
        band_fracs = {
            f'frac_{name}': (bp / total_power if np.isfinite(bp) and np.isfinite(total_power) and total_power > 0 else np.nan)
            for name, bp in band_powers.items()
        }
        p05, p50, p95 = _spectral_edges(f, p, probs=(0.05, 0.50, 0.95))
        cadence_ratios = {}
        for h in range(1, N_CADENCE_HARMONICS + 1):
            center = h * IMAGE_CADENCE_HZ
            if center <= PSD_MAX_HZ:
                cadence_ratios[f'cadence_h{h}_peak_over_bg'] = _peak_over_local_background(f, p, center)
        log_frac_psd, z_psd = _interpolate_row_normalized_logpsd(f, p, freq_grid)
        is_valid = bool(valid_rois[r] and finite_counts_total[r] >= MIN_VALID_SAMPLES and np.isfinite(total_power) and total_power > 0)

        metric_rows.append({
            'dmd': dmd,
            'roi_index': r,
            'roi_id': roi_ids[r],
            'valid_roi': is_valid,
            'sampling_rate_hz': fs,
            'downsample_factor': q,
            'downsampled_fs_hz': fs_ds,
            'n_psd_windows': windows_used,
            'psd_window_sec': PSD_WINDOW_SEC,
            'welch_segment_sec': WELCH_SEGMENT_SEC,
            'n_finite_samples': int(finite_counts_total[r]),
            'total_power': total_power,
            'spectral_centroid_hz': _spectral_centroid(f, p),
            'spectral_edge_05_hz': p05,
            'spectral_edge_50_hz': p50,
            'spectral_edge_95_hz': p95,
            'spectral_bandwidth_90_hz': p95 - p05 if np.isfinite(p95) and np.isfinite(p05) else np.nan,
            'spectral_entropy_band_norm': _normalized_entropy(list(band_fracs.values())),
            'psd_slope_0p1_10': _loglog_slope(f, p, 0.1, 10.0),
            'psd_slope_10_100': _loglog_slope(f, p, 10.0, 100.0),
            **band_fracs,
            **cadence_ratios,
        })
        psd_rows.append({
            'dmd': dmd,
            'roi_index': r,
            'roi_id': roi_ids[r],
            'valid_roi': is_valid,
            **{f'logpsd_frac_{freq:.5g}Hz': val for freq, val in zip(freq_grid, log_frac_psd)},
            **{f'logpsd_z_{freq:.5g}Hz': val for freq, val in zip(freq_grid, z_psd)},
        })

    metrics = pd.DataFrame(metric_rows)
    psd_grid = pd.DataFrame(psd_rows)
    raw = {'frequency_hz': f, 'psd': pxx_mean, 'roi_ids': roi_ids, 'valid_rois': valid_rois}
    return metrics, psd_grid, raw


def session_frequency_tables(asset):
    h5_path = resolve_trace_h5(asset)
    meta = _session_metadata(asset)
    metric_tables = []
    psd_tables = []
    raw_by_dmd = {}
    with h5py.File(h5_path, 'r') as h5:
        dmd_keys = sorted(k for k in h5.keys() if k.upper().startswith('DMD'))
        for dmd in dmd_keys:
            metrics, psd_grid, raw = estimate_dmd_frequency_metrics(h5[dmd], dmd=dmd)
            depth_col = 'dmd1_depth' if dmd.upper() == 'DMD1' else 'dmd2_depth' if dmd.upper() == 'DMD2' else None
            depth = meta.get(depth_col, np.nan) if depth_col else np.nan
            for tab in (metrics, psd_grid):
                tab.insert(0, 'trace_h5', str(h5_path))
                tab.insert(0, 'session_id', meta['session_id'])
                tab.insert(0, 'subject_id', meta['subject_id'])
                tab.insert(3, 'depth_um_below_pia', depth)
            metric_tables.append(metrics)
            psd_tables.append(psd_grid)
            raw_by_dmd[dmd] = raw
    return pd.concat(metric_tables, ignore_index=True), pd.concat(psd_tables, ignore_index=True), raw_by_dmd


## Plotting helpers

In [8]:

def _frequency_grid_from_psd_table(psd_grid, prefix='logpsd_z_'):
    cols = [c for c in psd_grid.columns if c.startswith(prefix)]
    freqs = np.asarray([float(c.replace(prefix, '').replace('Hz', '')) for c in cols], dtype=float)
    order = np.argsort(freqs)
    return [cols[i] for i in order], freqs[order]


def _sort_metric_rows(metrics, sort_by='spectral_centroid_hz'):
    df = metrics.query('valid_roi').copy()
    if sort_by not in df.columns:
        sort_by = 'spectral_centroid_hz'
    df['_dmd_order'] = df['dmd'].astype(str).str.extract(r'(\d+)').fillna(999).astype(int)
    df = df.sort_values(['_dmd_order', sort_by, 'session_id', 'roi_index']).drop(columns=['_dmd_order'])
    if MAX_HEATMAP_ROWS is not None and len(df) > MAX_HEATMAP_ROWS:
        take = np.unique(np.round(np.linspace(0, len(df) - 1, MAX_HEATMAP_ROWS)).astype(int))
        df = df.iloc[take].copy()
    return df


def plot_psd_heatmap(metrics, psd_grid, *, title=None, sort_by='spectral_centroid_hz', prefix='logpsd_z_'):
    cols, freqs = _frequency_grid_from_psd_table(psd_grid, prefix=prefix)
    order = _sort_metric_rows(metrics, sort_by=sort_by)
    key_cols = ['subject_id', 'session_id', 'dmd', 'roi_index', 'roi_id']
    merged = order[key_cols].merge(psd_grid[key_cols + cols], on=key_cols, how='left')
    Z = merged[cols].to_numpy(dtype=float)
    height = max(5.0, min(18.0, 2.8 + 0.035 * len(merged)))
    fig, ax = plt.subplots(figsize=(15, height))
    extent = [np.log10(freqs[0]), np.log10(freqs[-1]), len(merged), 0]
    im = ax.imshow(Z, aspect='auto', interpolation='nearest', extent=extent, vmin=-HEATMAP_ZLIM, vmax=HEATMAP_ZLIM)
    cbar = fig.colorbar(im, ax=ax, pad=0.01)
    cbar.set_label('Within-ROI log PSD z-score')
    ref_freqs = [0.01, 0.1, 1, 10, 100, 500]
    ax.set_xticks(np.log10(ref_freqs))
    ax.set_xticklabels([str(f) for f in ref_freqs])
    ax.set_xlim(np.log10(freqs[0]), np.log10(freqs[-1]))
    ax.set_xlabel('Frequency (Hz, log scale)')
    ax.set_ylabel('ROIs grouped by DMD')
    for h in range(1, N_CADENCE_HARMONICS + 1):
        f0 = h * IMAGE_CADENCE_HZ
        if freqs[0] <= f0 <= freqs[-1]:
            ax.axvline(np.log10(f0), linestyle='--' if h == 1 else ':', linewidth=1.0 if h == 1 else 0.7, alpha=0.7)
            if h <= 4:
                ax.text(np.log10(f0), 0.995, f'{h}× image', rotation=90, va='top', ha='right', fontsize=9, transform=ax.get_xaxis_transform())
    y0 = 0
    for dmd, sub in merged.groupby('dmd', sort=False):
        y1 = y0 + len(sub)
        ax.axhline(y1, linewidth=1.0, alpha=0.6)
        ax.text(np.log10(freqs[0]) - 0.04 * (np.log10(freqs[-1]) - np.log10(freqs[0])), (y0 + y1) / 2, f'{dmd}\n(n={len(sub)})', va='center', ha='right', fontsize=10)
        y0 = y1
    ax.set_title(title or 'ROI × frequency PSD heatmap')
    fig.tight_layout()
    return fig, ax


def plot_bandpower_heatmap(metrics, *, title=None, sort_by='spectral_centroid_hz'):
    band_cols = [f'frac_{name}' for name in FREQUENCY_BANDS]
    labels = [BAND_LABELS[name] for name in FREQUENCY_BANDS]
    order = _sort_metric_rows(metrics, sort_by=sort_by)
    Z = order[band_cols].to_numpy(dtype=float)
    height = max(5.0, min(18.0, 2.8 + 0.035 * len(order)))
    fig, ax = plt.subplots(figsize=(9, height))
    im = ax.imshow(Z, aspect='auto', interpolation='nearest', vmin=0, vmax=np.nanpercentile(Z, 98))
    cbar = fig.colorbar(im, ax=ax, pad=0.01)
    cbar.set_label('Fraction of 0.01-500 Hz power')
    ax.set_xticks(np.arange(len(labels)))
    ax.set_xticklabels(labels, rotation=30, ha='right')
    ax.set_ylabel('ROIs grouped by DMD')
    y0 = 0
    for dmd, sub in order.groupby('dmd', sort=False):
        y1 = y0 + len(sub)
        ax.axhline(y1 - 0.5, linewidth=1.0, alpha=0.6)
        ax.text(-0.65, (y0 + y1 - 1) / 2, f'{dmd}\n(n={len(sub)})', va='center', ha='right', fontsize=10)
        y0 = y1
    ax.set_title(title or 'ROI × bandpower fraction heatmap')
    fig.tight_layout()
    return fig, ax


def plot_bandpower_distributions(metrics):
    df = metrics.query('valid_roi').copy()
    band_names = list(FREQUENCY_BANDS.keys())
    band_cols = [f'frac_{name}' for name in band_names]
    labels = [BAND_LABELS[name].replace('\n', ' ') for name in band_names]
    fig, axes = plt.subplots(1, len(band_cols), figsize=(3.4 * len(band_cols), 5), sharey=True)
    if len(band_cols) == 1:
        axes = [axes]
    rng = np.random.default_rng(0)
    for ax, col, label in zip(axes, band_cols, labels):
        groups = [sub[col].dropna().values for _, sub in df.groupby('dmd')]
        group_labels = [str(k) for k, _ in df.groupby('dmd')]
        if all(len(g) > 0 for g in groups):
            ax.boxplot(groups, labels=group_labels, showfliers=False)
            for xi, vals in enumerate(groups, start=1):
                jitter = (rng.random(len(vals)) - 0.5) * 0.18
                ax.plot(np.full(len(vals), xi) + jitter, vals, 'o', ms=3, alpha=0.35)
        ax.set_title(label)
        ax.set_xlabel('DMD')
    axes[0].set_ylabel('Fraction of 0.01-500 Hz power')
    fig.suptitle('Bandpower fractions by DMD', y=1.04)
    fig.tight_layout()
    return fig, axes


def plot_cadence_enrichment(metrics):
    df = metrics.query('valid_roi').copy()
    cadence_cols = [c for c in df.columns if c.startswith('cadence_h') and c.endswith('_peak_over_bg')]
    fig, axes = plt.subplots(1, min(4, len(cadence_cols)), figsize=(4.2 * min(4, len(cadence_cols)), 4.5), sharey=True)
    if not isinstance(axes, np.ndarray):
        axes = np.asarray([axes])
    for ax, col in zip(axes, cadence_cols[:len(axes)]):
        h = col.split('_')[1].replace('h', '')
        groups = [sub[col].replace([np.inf, -np.inf], np.nan).dropna().values for _, sub in df.groupby('dmd')]
        labels = [str(k) for k, _ in df.groupby('dmd')]
        ax.boxplot(groups, labels=labels, showfliers=False)
        ax.axhline(1.0, linewidth=1.0, alpha=0.6)
        ax.set_yscale('log')
        ax.set_title(f'{h}× image cadence\n{float(h) * IMAGE_CADENCE_HZ:.2f} Hz')
        ax.set_xlabel('DMD')
    axes[0].set_ylabel('PSD peak / local background')
    fig.suptitle('Image-cadence enrichment by ROI', y=1.05)
    fig.tight_layout()
    return fig, axes


## 1. Example session: heatmaps and frequency phenotypes

In [9]:

example_asset = assets[EXAMPLE_ASSET_INDEX]
example_metrics, example_psd_grid, example_raw_psd = session_frequency_tables(example_asset)
example_valid = example_metrics.query('valid_roi').copy()

print(f'Example session: {example_asset.session_id}')
print(resolve_trace_h5(example_asset))
display(example_valid[[
    'dmd', 'roi_id', 'depth_um_below_pia', 'spectral_centroid_hz',
    'spectral_entropy_band_norm', 'spectral_bandwidth_90_hz',
    'frac_very_slow_0p01_0p1', 'frac_task_0p5_2', 'frac_fast_10_100', 'cadence_h1_peak_over_bg'
]].sort_values(['dmd', 'spectral_centroid_hz']).head(40))

fig, ax = plot_psd_heatmap(
    example_metrics,
    example_psd_grid,
    title=f'Example session {example_asset.session_id}: normalized PSD profiles',
    sort_by='spectral_centroid_hz',
)
if SAVE_FIGURES:
    fig.savefig(SAVE_PATH / f'{today_str}_example_session_psd_heatmap.png', bbox_inches='tight', dpi=300)
plt.show()

fig, ax = plot_bandpower_heatmap(
    example_metrics,
    title=f'Example session {example_asset.session_id}: bandpower fractions',
)
if SAVE_FIGURES:
    fig.savefig(SAVE_PATH / f'{today_str}_example_session_bandpower_heatmap.png', bbox_inches='tight', dpi=300)
plt.show()


Example session: 826031_2026-01-30_15-04-02
\\allen\aind\scratch\ophys\Andrew\VIP_synaptic_dynamics\ASAP7\826031\826031_2026-01-30_15-04-02\analysis\derived\voltage\voltage_session_traces_dff_robust_f0_trial.h5


,dmd,roi_id,depth_um_below_pia,spectral_centroid_hz,spectral_entropy_band_norm,spectral_bandwidth_90_hz,frac_very_slow_0p01_0p1,frac_task_0p5_2,frac_fast_10_100,cadence_h1_peak_over_bg
4,DMD1,DMD1_roi0004,25,1.597431,0.731148,3.538080,0.431328,0.175947,0.018367,28.395587
0,DMD1,DMD1_roi0000,25,1.755001,0.668244,3.287202,0.487730,0.087956,0.021406,8.116747
1,DMD1,DMD1_roi0001,25,3.264221,0.760907,6.538039,0.423294,0.165252,0.033196,22.434101
2,DMD1,DMD1_roi0002,25,4.548483,0.808459,10.100933,0.377888,0.246390,0.039050,38.663822
3,DMD1,DMD1_roi0003,25,4.683284,0.808987,8.565617,0.371671,0.255951,0.030061,32.126592
11,DMD1,DMD1_roi0011,25,5.192932,0.828290,16.573996,0.361659,0.202694,0.046901,12.058382
13,DMD1,DMD1_roi0013,25,6.152445,0.822285,24.053766,0.244410,0.412957,0.056646,64.662539
14,DMD1,DMD1_roi0014,25,6.377693,0.857878,28.175461,0.311715,0.281368,0.072728,47.776329
6,DMD1,DMD1_roi0006,25,8.975920,0.910118,51.050339,0.264570,0.249376,0.103915,11.124372
10,DMD1,DMD1_roi0010,25,10.153601,0.856494,63.747599,0.170080,0.424594,0.128847,47.340428


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [10]:
fig,ax=plt.subplots()
freq = example_raw_psd['DMD1']['frequency_hz']
data = example_raw_psd['DMD1']['psd'][0]

ax.plot(freq,data)

ax.loglog()

<IPython.core.display.Javascript object>

[]

In [11]:
freq

array([0.00000000e+00, 1.56250598e-02, 3.12501196e-02, ...,
       4.99955038e+02, 4.99970663e+02, 4.99986288e+02])

In [12]:

def pick_frequency_phenotype_rois(metrics):
    df = metrics.query('valid_roi').copy()
    if df.empty:
        raise ValueError('No valid ROIs available')
    picks = {}
    slow_score = df['frac_very_slow_0p01_0p1'].fillna(0) + df['frac_slow_0p1_0p5'].fillna(0)
    picks['slow-dominated'] = df.loc[slow_score.idxmax()]
    task_score = df['frac_task_0p5_2'].fillna(0)
    if 'cadence_h1_peak_over_bg' in df:
        cadence_bonus = np.log10(df['cadence_h1_peak_over_bg'].replace([np.inf, -np.inf], np.nan).fillna(1).clip(lower=1))
        task_score = task_score + 0.05 * cadence_bonus
    picks['task-band enriched'] = df.loc[task_score.idxmax()]
    broad_score = df['spectral_entropy_band_norm'].fillna(0) + df['frac_fast_10_100'].fillna(0)
    picks['broadband / fast'] = df.loc[broad_score.idxmax()]
    return picks


def _read_roi_trace_for_display(asset, dmd, roi_index, trace_start_sec=None, duration_sec=TRACE_DURATION_SEC):
    h5_path = resolve_trace_h5(asset)
    with h5py.File(h5_path, 'r') as h5:
        g = h5[dmd]
        t = np.asarray(g['timebase_sec'][:], dtype=np.float64)
        fs = _infer_sampling_rate_hz(t)
        ds = g[SIGNAL]
        layout = _trace_layout(ds, len(t))
        t0 = float(t[0] + 30.0) if trace_start_sec is None else float(trace_start_sec)
        t1 = t0 + duration_sec
        i0 = int(np.searchsorted(t, t0, side='left'))
        i1 = int(np.searchsorted(t, t1, side='right'))
        X = _read_trace_block(ds, i0, i1, layout)
        x = X[int(roi_index)]
        tt = t[i0:i1]
    finite = np.isfinite(x)
    x = x.copy()
    x[~finite] = np.nanmedian(x[finite]) if finite.any() else 0.0
    q = max(1, int(np.floor(fs / TRACE_DISPLAY_FS_HZ)))
    if q > 1 and x.size > q * 4:
        x2 = signal.resample_poly(x[np.newaxis, :], up=1, down=q, axis=1, padtype='line')[0]
        t2 = np.linspace(tt[0], tt[-1], x2.size)
    else:
        x2, t2 = x, tt
    return t2 - t2[0], x2


def plot_example_frequency_phenotypes(asset, metrics, psd_grid):
    picks = pick_frequency_phenotype_rois(metrics)
    frac_cols, freqs = _frequency_grid_from_psd_table(psd_grid, prefix='logpsd_frac_')
    key_cols = ['subject_id', 'session_id', 'dmd', 'roi_index', 'roi_id']
    merged = metrics[key_cols].merge(psd_grid[key_cols + frac_cols], on=key_cols, how='left')
    fig, axes = plt.subplots(len(picks), 2, figsize=(13, 3.4 * len(picks)))
    if len(picks) == 1:
        axes = axes[np.newaxis, :]
    for row_i, (label, row) in enumerate(picks.items()):
        dmd = row['dmd']
        roi_index = int(row['roi_index'])
        roi_id = row['roi_id']
        tt, xx = _read_roi_trace_for_display(asset, dmd, roi_index, TRACE_START_SEC, TRACE_DURATION_SEC)
        ax_trace, ax_psd = axes[row_i]
        ax_trace.plot(tt, xx, lw=0.8)
        ax_trace.axhline(0, lw=0.7, alpha=0.5)
        ax_trace.set_xlabel('Time (s)')
        ax_trace.set_ylabel('dF/F')
        ax_trace.set_title(f'{label}: {dmd} ROI {roi_id}')
        m = (merged['dmd'].eq(dmd) & merged['roi_index'].astype(int).eq(roi_index) & merged['roi_id'].astype(str).eq(str(roi_id)))
        if m.any():
            psd_vals = merged.loc[m, frac_cols].iloc[0].to_numpy(dtype=float)
            ax_psd.plot(freqs, 10 ** psd_vals, lw=1.4)
        for h in range(1, N_CADENCE_HARMONICS + 1):
            f0 = h * IMAGE_CADENCE_HZ
            if freqs[0] <= f0 <= freqs[-1]:
                ax_psd.axvline(f0, linestyle='--' if h == 1 else ':', linewidth=0.8, alpha=0.5)
        ax_psd.set_xscale('log')
        ax_psd.set_yscale('log')
        ax_psd.set_xlim(FREQ_MIN_PLOT, FREQ_MAX_PLOT)
        ax_psd.set_xlabel('Frequency (Hz)')
        ax_psd.set_ylabel('Fractional PSD density')
        ax_psd.set_title(f'centroid={row["spectral_centroid_hz"]:.2g} Hz, entropy={row["spectral_entropy_band_norm"]:.2f}')
        ax_psd.grid(True, which='both', alpha=0.25)
    fig.suptitle(f'Example frequency phenotypes — {asset.session_id}', y=1.02)
    fig.tight_layout()
    return fig, axes

fig, axes = plot_example_frequency_phenotypes(example_asset, example_metrics, example_psd_grid)
if SAVE_FIGURES:
    fig.savefig(SAVE_PATH / f'{today_str}_example_frequency_phenotype_rois.png', bbox_inches='tight', dpi=300)
plt.show()


<IPython.core.display.Javascript object>

## 2. All sessions: pooled ROI frequency metrics

In [ ]:

all_metric_tables = []
all_psd_tables = []
failures = []

for i, asset in enumerate(assets, start=1):
    print(f'[{i:02d}/{len(assets):02d}] {asset.session_id}')
    try:
        m, p, _ = session_frequency_tables(asset)
        all_metric_tables.append(m)
        all_psd_tables.append(p)
    except Exception as exc:
        warnings.warn(f'Failed {asset.session_id}: {exc}')
        failures.append({'session_id': str(getattr(asset, 'session_id', 'unknown')), 'error': repr(exc)})

if not all_metric_tables:
    raise RuntimeError('No session frequency metrics were successfully calculated.')

freq_metrics_df = pd.concat(all_metric_tables, ignore_index=True)
psd_grid_df = pd.concat(all_psd_tables, ignore_index=True)
valid_freq_df = freq_metrics_df.query('valid_roi').copy()
valid_psd_grid_df = psd_grid_df.query('valid_roi').copy()

print(f'Calculated {len(freq_metrics_df):,} ROI frequency records from {freq_metrics_df.session_id.nunique()} sessions.')
print(f'Valid ROI metrics: {len(valid_freq_df):,}')
if failures:
    display(pd.DataFrame(failures))

display(valid_freq_df.groupby('dmd')[[
    'spectral_centroid_hz', 'spectral_entropy_band_norm', 'spectral_bandwidth_90_hz',
    'frac_very_slow_0p01_0p1', 'frac_task_0p5_2', 'frac_fast_10_100', 'cadence_h1_peak_over_bg'
]].agg(['count', 'median', 'mean', 'std']))

if SAVE_TABLES:
    metrics_path = SAVE_PATH / f'{today_str}_ASAP7_frequency_roi_metrics_all_sessions.csv'
    psd_grid_path = SAVE_PATH / f'{today_str}_ASAP7_frequency_psd_loggrid_all_sessions.csv'
    freq_metrics_df.to_csv(metrics_path, index=False)
    psd_grid_df.to_csv(psd_grid_path, index=False)
    print(f'Saved: {metrics_path}')
    print(f'Saved: {psd_grid_path}')


[01/19] 826031_2026-01-30_15-04-02
[02/19] 826031_2026-02-01_11-01-50
[03/19] 826031_2026-02-02_10-23-53
[04/19] 826031_2026-02-03_14-21-45
[05/19] 826031_2026-02-04_12-15-34
[06/19] 826031_2026-02-05_09-28-56
[07/19] 826031_2026-02-06_10-21-20
[08/19] 826031_2026-02-10_10-48-51
[09/19] 826031_2026-02-11_12-42-21
[10/19] 826031_2026-02-12_07-43-37
[11/19] 826032_2026-01-29_12-59-37


In [ ]:

fig, ax = plot_psd_heatmap(
    valid_freq_df,
    valid_psd_grid_df,
    title='All sessions: normalized PSD profiles across VIP dendritic ROIs',
    sort_by='spectral_centroid_hz',
)
if SAVE_FIGURES:
    fig.savefig(SAVE_PATH / f'{today_str}_all_sessions_psd_heatmap.png', bbox_inches='tight', dpi=300)
plt.show()

fig, ax = plot_bandpower_heatmap(
    valid_freq_df,
    title='All sessions: bandpower fractions across VIP dendritic ROIs',
    sort_by='spectral_centroid_hz',
)
if SAVE_FIGURES:
    fig.savefig(SAVE_PATH / f'{today_str}_all_sessions_bandpower_heatmap.png', bbox_inches='tight', dpi=300)
plt.show()

fig, axes = plot_bandpower_distributions(valid_freq_df)
if SAVE_FIGURES:
    fig.savefig(SAVE_PATH / f'{today_str}_bandpower_fraction_by_dmd.png', bbox_inches='tight', dpi=300)
plt.show()

fig, axes = plot_cadence_enrichment(valid_freq_df)
if SAVE_FIGURES:
    fig.savefig(SAVE_PATH / f'{today_str}_image_cadence_enrichment_by_dmd.png', bbox_inches='tight', dpi=300)
plt.show()


## 3. Frequency phenotype space

In [ ]:

def _find_latest_snr_csv():
    if SNR_METRICS_CSV is not None:
        p = Path(SNR_METRICS_CSV)
        return p if p.exists() else None
    candidates = sorted(SAVE_PATH.glob('*ASAP7_dynamic_snr_all_sessions.csv'))
    return candidates[-1] if candidates else None


def merge_optional_snr(freq_df):
    snr_csv = _find_latest_snr_csv()
    if snr_csv is None:
        print('No companion SNR CSV found; phenotype scatter will use uniform marker size.')
        out = freq_df.copy()
        out['snr_dynamic'] = np.nan
        out['snr_db'] = np.nan
        return out
    snr = pd.read_csv(snr_csv)
    key_cols = ['subject_id', 'session_id', 'dmd', 'roi_id']
    if not all(c in snr.columns and c in freq_df.columns for c in key_cols):
        print(f'Found SNR CSV but could not merge on expected keys: {snr_csv}')
        out = freq_df.copy()
        out['snr_dynamic'] = np.nan
        out['snr_db'] = np.nan
        return out
    keep = key_cols + [c for c in ['snr_dynamic', 'snr_db', 'amplitude_p95_p05', 'noise_sigma_dff'] if c in snr.columns]
    out = freq_df.merge(snr[keep].drop_duplicates(key_cols), on=key_cols, how='left')
    print(f'Merged SNR metrics from: {snr_csv}')
    return out


def plot_frequency_phenotype_space(df):
    plot_df = df.query('valid_roi').copy()
    fig, ax = plt.subplots(figsize=(8, 6))
    if 'snr_dynamic' in plot_df and plot_df['snr_dynamic'].notna().any():
        snr = plot_df['snr_dynamic'].to_numpy(dtype=float)
        lo, hi = np.nanpercentile(snr, [5, 95])
        size = 25 + 95 * np.clip((snr - lo) / (hi - lo + 1e-12), 0, 1)
    else:
        size = np.full(len(plot_df), 45.0)
    for dmd, sub in plot_df.groupby('dmd'):
        loc = plot_df.index.get_indexer(sub.index)
        ax.scatter(sub['spectral_centroid_hz'], sub['spectral_entropy_band_norm'], s=size[loc], alpha=0.65, label=dmd)
    ax.set_xscale('log')
    ax.set_xlabel('Spectral centroid (Hz)')
    ax.set_ylabel('Bandpower spectral entropy (0-1)')
    ax.set_title('Frequency phenotype space\nsize = dynamic SNR if companion SNR CSV is available')
    ax.legend(title='DMD', frameon=False)
    ax.grid(True, which='both', alpha=0.25)
    fig.tight_layout()
    return fig, ax

freq_with_snr = merge_optional_snr(valid_freq_df)
fig, ax = plot_frequency_phenotype_space(freq_with_snr)
if SAVE_FIGURES:
    fig.savefig(SAVE_PATH / f'{today_str}_frequency_phenotype_centroid_entropy.png', bbox_inches='tight', dpi=300)
plt.show()


## 4. Session-level summaries

In [ ]:

metric_summary_cols = [
    'spectral_centroid_hz',
    'spectral_entropy_band_norm',
    'spectral_bandwidth_90_hz',
    'psd_slope_0p1_10',
    'psd_slope_10_100',
    'cadence_h1_peak_over_bg',
] + [f'frac_{name}' for name in FREQUENCY_BANDS]

session_level_freq = (
    valid_freq_df
    .groupby(['subject_id', 'session_id', 'dmd', 'depth_um_below_pia'], as_index=False)[metric_summary_cols]
    .median()
)
n_roi = (
    valid_freq_df
    .groupby(['subject_id', 'session_id', 'dmd', 'depth_um_below_pia'])
    .size()
    .reset_index(name='n_valid_roi')
)
session_level_freq = session_level_freq.merge(n_roi, on=['subject_id', 'session_id', 'dmd', 'depth_um_below_pia'], how='left')
display(session_level_freq)

if SAVE_TABLES:
    session_path = SAVE_PATH / f'{today_str}_ASAP7_frequency_session_level_summary.csv'
    session_level_freq.to_csv(session_path, index=False)
    print(f'Saved: {session_path}')

fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
ycols = ['spectral_centroid_hz', 'spectral_entropy_band_norm', 'frac_task_0p5_2']
titles = ['Median spectral centroid', 'Median spectral entropy', 'Median task-band fraction']
ylabels = ['Hz', '0-1', 'Fraction of 0.01-500 Hz power']
rng = np.random.default_rng(1)
for ax, ycol, title, ylabel in zip(axes, ycols, titles, ylabels):
    for dmd, sub in session_level_freq.groupby('dmd'):
        xval = int(str(dmd).replace('DMD', '')) if str(dmd).replace('DMD', '').isdigit() else np.nan
        jitter = (rng.random(len(sub)) - 0.5) * 0.08
        ax.plot(np.full(len(sub), xval) + jitter, sub[ycol], 'o', alpha=0.75, label=dmd if ycol == ycols[0] else None)
    ax.set_xticks([1, 2])
    ax.set_xticklabels(['DMD1', 'DMD2'])
    if ycol == 'spectral_centroid_hz':
        ax.set_yscale('log')
    ax.set_title(title)
    ax.set_ylabel(ylabel)
    ax.grid(True, axis='y', alpha=0.25)
axes[0].legend(frameon=False)
fig.suptitle('Session-level frequency summaries by DMD', y=1.03)
fig.tight_layout()
if SAVE_FIGURES:
    fig.savefig(SAVE_PATH / f'{today_str}_session_level_frequency_summary_by_dmd.png', bbox_inches='tight', dpi=300)
plt.show()



## Interpretation notes for slide 6

For a first-pass lab-meeting slide, I would use the following hierarchy of claims:

1. **Safe descriptive claim:** VIP dendritic `dF/F` contains measurable power across slow, task-timescale, intermediate, and faster bands.
2. **ROI heterogeneity claim:** individual dendrites differ in their spectral profiles; some are dominated by slow fluctuations, some show stronger 0.5-2 Hz/task-cadence structure, and some have broader or faster content.
3. **DMD/depth claim:** make this from session-level summaries, not raw ROI counts, because ROIs are nested within sessions and are not independent samples across animals.
4. **Cadence caution:** a peak at ~1.33 Hz or its harmonics is not automatically “visual response.” It can reflect true image-locked voltage responses, event-alignment leakage, periodic expectation/ramping, running entrainment, optical/electrical artifacts, or processing/F0 interactions. Treat cadence enrichment here as characterization/QC; mechanistic attribution belongs in the later shared-signal and event-model analyses.
5. **Noise-floor caution:** 100-500 Hz power is useful as a high-frequency reference, but it should not be assumed pure noise unless validated with blank/LED-off/control recordings or with within-session comparisons to robust SNR estimates.

Slide takeaway: **ASAP7 dendritic dF/F contains slow, task-timescale, and faster components whose relative contributions vary across ROIs. Shared structure across these bands is analyzed later.**
